# AI-Lake + Apache Spark

Spark reads AI-Lake tables as standard Iceberg tables — no AI-Lake plugin required
for tabular queries. This notebook uses **PySpark local[*]** mode, which runs
entirely inside the Jupyter container (no separate Spark cluster needed).

**What this notebook covers:**
1. Read AI-Lake table as plain Parquet
2. Read via Iceberg `HadoopCatalog` (SQL interface)
3. Filtered scans and aggregations
4. Schema inspection (`embedding` column shows as `binary`)

In [ ]:
import os
import pathlib
from pyspark.sql import SparkSession

TABLE_PATH = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
parquet_files = sorted(str(p) for p in pathlib.Path(TABLE_PATH).rglob('*.parquet'))
print(f'Table path   : {TABLE_PATH}')
print(f'Parquet files: {len(parquet_files)}')

## 1. Create SparkSession

In [ ]:
import os
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

spark = (
    SparkSession.builder
    .appName('ailake-demo')
    .master('local[2]')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} — local[2] mode')

## 2. Direct Parquet read

In [ ]:
df = spark.read.parquet(*parquet_files)
print(f'Rows: {df.count()}')
print('Schema:')
df.printSchema()

In [ ]:
# embedding shows as binary — HNSW footer is invisible to Spark
df.select('text').show(5, truncate=80)

## 3. Iceberg HadoopCatalog (SQL interface)

In [ ]:
import os
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

# New session with Iceberg extensions
spark.stop()
spark_iceberg = (
    SparkSession.builder
    .appName('ailake-iceberg-demo')
    .master('local[2]')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '2')
    .config(
        'spark.jars.packages',
        'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2',
    )
    .config(
        'spark.sql.extensions',
        'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions',
    )
    .config('spark.sql.catalog.ailake', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.ailake.type', 'hadoop')
    .config('spark.sql.catalog.ailake.warehouse', f'file://{TABLE_PATH}')
    .getOrCreate()
)
spark_iceberg.sparkContext.setLogLevel('ERROR')
print('Iceberg SparkSession ready')

In [ ]:
# SQL query via Iceberg catalog — identical to querying any Iceberg table
count = spark_iceberg.sql(
    'SELECT count(*) AS n FROM ailake.default.table'
).collect()[0][0]
print(f'Iceberg SQL count: {count}')

In [ ]:
spark_iceberg.sql("""
    SELECT text
    FROM ailake.default.table
    WHERE text LIKE '%vector search%'
    LIMIT 5
""").show(truncate=90)

## 4. Aggregations

In [ ]:
spark_iceberg.sql("""
    SELECT
        count(*)               AS total_rows,
        avg(length(text))      AS avg_text_len,
        avg(length(embedding)) AS avg_emb_bytes
    FROM ailake.default.table
""").show()

## 5. Iceberg snapshot history

In [ ]:
spark_iceberg.sql(
    'SELECT snapshot_id, committed_at, operation FROM ailake.default.table.history'
).show(truncate=False)

In [ ]:
spark_iceberg.stop()
print('SparkSession stopped.')

## Key takeaway

Spark reads AI-Lake tables as **standard Iceberg** — the `HadoopCatalog` discovers
files from the manifest, reads Parquet row groups normally, and returns the `embedding`
column as `binary`. No AI-Lake plugin is needed for tabular queries.

To enable vector search from Spark, install the AI-Lake Spark plugin
(see `docs/specs/JVM_PLUGINS.md`).